# Nonlinear Supervised Hierarchical Rao-Style PC-CNN — V3

## Architecture

```text
Image ⇄ r1 ⇄ r2 ⇄ r3 → Linear → Softmax → Class
```

- `r1`: 32 × 32 × 32
- `r2`: 128 × 16 × 16
- `r3`: 256 × 8 × 8
- `tanh` is used as the nonlinear representation function.
- Classification uses the full **free `r3`** representation.
- Classification error nudges **only `r3`**.
- Lower representations are influenced through the predictive hierarchy.
- `U1/U2/U3` use local Rao-style predictive learning.
- The classifier uses a local supervised update from free `r3`.
- **No autograd, no backpropagation, no optimizer.**


In [1]:
import copy
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from collections import defaultdict
from torch.utils.data import DataLoader, Subset

import src.data_cleaning_and_preprocessing._model_ready_dataset as model_ready_dataset
from src.dataset.tiny_imagenet_dataset import TinyImageNetLoader
from archive.pc_cnn.py_file.pc_supervised_rao_nonlinear_v1 import HierarchicalPCCNN

In [2]:
SEED = 42
BATCH_SIZE = 16
IMAGE_SIZE = 64

PC_EPOCHS = 5

NUM_CLASSES = 20
TRAIN_IMAGES_PER_CLASS = 200
VAL_IMAGES_PER_CLASS = 50
TEST_IMAGES_PER_CLASS = 50

INFERENCE_STEPS = 100
SUPERVISED_NUDGE_STEPS = 20

SIGMA_CLASS_SQ = 1.0
CLASS_LEARNING_RATE = 0.01

In [3]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print("Device:", device)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available() and hasattr(torch.mps, "manual_seed"):
    torch.mps.manual_seed(SEED)

Device: mps


In [4]:
cleaned_tiny_imagenet = model_ready_dataset.ModelReadyDatasetLoader(
    TinyImageNetLoader(seed=SEED),
    image_size=IMAGE_SIZE
).load()

train_dataset = cleaned_tiny_imagenet[0]
test_dataset = cleaned_tiny_imagenet[1]

print("Full training dataset:", len(train_dataset))
print("Full test dataset:", len(test_dataset))

Full training dataset: 99953
Full test dataset: 9993


In [5]:
selected_classes = list(range(NUM_CLASSES))

print("Selected classes:", selected_classes)
print("Number of selected classes:", len(selected_classes))

Selected classes: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]
Number of selected classes: 20


In [6]:
class_indices = defaultdict(list)

for i in range(len(train_dataset)):
    _, label = train_dataset[i]
    label = int(label)

    if label in selected_classes:
        class_indices[label].append(i)

In [7]:
split_generator = torch.Generator()
split_generator.manual_seed(SEED)

train_indices = []
val_indices = []

for class_id in selected_classes:
    available_indices = class_indices[class_id]
    required_images = TRAIN_IMAGES_PER_CLASS + VAL_IMAGES_PER_CLASS

    if len(available_indices) < required_images:
        raise ValueError(
            f"Class {class_id} has only {len(available_indices)} images, "
            f"but {required_images} are required."
        )

    permutation = torch.randperm(
        len(available_indices),
        generator=split_generator
    )

    train_selection = permutation[:TRAIN_IMAGES_PER_CLASS]
    val_selection = permutation[
        TRAIN_IMAGES_PER_CLASS:
        TRAIN_IMAGES_PER_CLASS + VAL_IMAGES_PER_CLASS
    ]

    train_selected = [
        available_indices[i]
        for i in train_selection.tolist()
    ]

    val_selected = [
        available_indices[i]
        for i in val_selection.tolist()
    ]

    train_indices.extend(train_selected)
    val_indices.extend(val_selected)

train_subset = Subset(train_dataset, train_indices)
val_subset = Subset(train_dataset, val_indices)

overlap = set(train_indices).intersection(
    set(val_indices)
)

print("Training images:", len(train_subset))
print("Validation images:", len(val_subset))
print("Train/Validation overlap:", len(overlap))

assert len(overlap) == 0

Training images: 4000
Validation images: 1000
Train/Validation overlap: 0


In [8]:
train_loader_generator = torch.Generator()
train_loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_loader_generator
)

train_eval_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Training evaluation batches:", len(train_eval_loader))
print("Validation batches:", len(val_loader))

Training batches: 250
Training evaluation batches: 250
Validation batches: 63


In [9]:
test_class_indices = defaultdict(list)

for i in range(len(test_dataset)):
    _, label = test_dataset[i]
    label = int(label)

    if label in selected_classes:
        test_class_indices[label].append(i)

test_generator = torch.Generator()
test_generator.manual_seed(SEED)

test_indices = []

for class_id in selected_classes:
    available_indices = test_class_indices[class_id]

    if len(available_indices) < TEST_IMAGES_PER_CLASS:
        raise ValueError(
            f"Test class {class_id} has only "
            f"{len(available_indices)} images, "
            f"but {TEST_IMAGES_PER_CLASS} were requested."
        )

    permutation = torch.randperm(
        len(available_indices),
        generator=test_generator
    )

    selection = permutation[:TEST_IMAGES_PER_CLASS]

    selected = [
        available_indices[i]
        for i in selection.tolist()
    ]

    test_indices.extend(selected)

test_subset = Subset(test_dataset, test_indices)

test_loader = DataLoader(
    test_subset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Final test images:", len(test_subset))

Final test images: 1000


In [10]:
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if torch.backends.mps.is_available() and hasattr(torch.mps, "manual_seed"):
    torch.mps.manual_seed(SEED)

model = HierarchicalPCCNN(
    num_classes=NUM_CLASSES,
    input_size=IMAGE_SIZE,
    inference_steps=INFERENCE_STEPS,
    supervised_nudge_steps=SUPERVISED_NUDGE_STEPS,
    convergence_tolerance=1e-5,
    sigma_class_sq=SIGMA_CLASS_SQ,
    class_learning_rate=CLASS_LEARNING_RATE
).to(device)

print(model)

print("sigma_bu_sq:", model.sigma_bu_sq)
print("sigma_td_sq:", model.sigma_td_sq)
print("sigma_class_sq:", model.sigma_class_sq)
print("class_learning_rate:", model.class_learning_rate)
print("free inference steps:", model.inference_steps)
print("supervised nudge steps:", model.supervised_nudge_steps)
print("k2 decay interval:", model.k2_decay_interval)
print("inference_dt:", model.inference_dt)
print("weight_dt:", model.weight_dt)

HierarchicalPCCNN(
  (level1): ModuleDict(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (deconv): ConvTranspose2d(32, 3, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), output_padding=(1, 1), bias=False)
  )
  (level2): ModuleDict(
    (conv): Conv2d(32, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (deconv): ConvTranspose2d(128, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), output_padding=(1, 1), bias=False)
  )
  (level3): ModuleDict(
    (conv): Conv2d(128, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (deconv): ConvTranspose2d(256, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), output_padding=(1, 1), bias=False)
  )
  (classifier): Linear(in_features=16384, out_features=20, bias=True)
)
sigma_bu_sq: 1.0
sigma_td_sq: 10.0
sigma_class_sq: 1.0
class_learning_rate: 0.01
free inference steps: 100
supervised nudge steps: 20
k2 decay interval: 2500
inference_dt: 0.1
wei

In [11]:
sample_image, _ = train_subset[0]
sample_image = sample_image.to(device)

with torch.no_grad():
    r1, r2, r3 = model.initialize_states(sample_image)

print("r1 shape:", tuple(r1.shape))
print("r2 shape:", tuple(r2.shape))
print("r3 shape:", tuple(r3.shape))
print("Classifier input features:", model.classifier.in_features)
print("Expected classifier input:", 256 * model.r3_size * model.r3_size)

r1 shape: (1, 32, 32, 32)
r2 shape: (1, 128, 16, 16)
r3 shape: (1, 256, 8, 8)
Classifier input features: 16384
Expected classifier input: 16384


In [12]:
total_params = sum(
    parameter.numel()
    for parameter in model.parameters()
)

autograd_params = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Total unique parameters:", total_params)
print("Autograd-enabled parameters:", autograd_params)

assert autograd_params == 0

Total unique parameters: 660340
Autograd-enabled parameters: 0


In [13]:
initial_u1 = model.level1["conv"].weight.detach().cpu().clone()
initial_u2 = model.level2["conv"].weight.detach().cpu().clone()
initial_u3 = model.level3["conv"].weight.detach().cpu().clone()
initial_wc = model.classifier.weight.detach().cpu().clone()
initial_bc = model.classifier.bias.detach().cpu().clone()

print("Initial U1 norm:", initial_u1.norm().item())
print("Initial U2 norm:", initial_u2.norm().item())
print("Initial U3 norm:", initial_u3.norm().item())
print("Initial Wc norm:", initial_wc.norm().item())
print("Initial bc:", initial_bc)

print("Initial training_input_count:", model.training_input_count)
print("Initial k2:", model.k2)

Initial U1 norm: 0.30018362402915955
Initial U2 norm: 1.9215914011001587
Initial U3 norm: 5.42879581451416
Initial Wc norm: 5.7302069664001465
Initial bc: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])
Initial training_input_count: 0
Initial k2: 1.0


In [14]:
@torch.no_grad()
def evaluate_model(model, data_loader, device, name=""):
    total_loss = 0.0
    total_correct = 0
    total_images = 0
    total_steps = 0
    total_converged = 0

    model.eval()

    for images, labels in tqdm(
        data_loader,
        desc=f"Evaluation Nonlinear Rao PC-CNN - {name}"
    ):
    # for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(
            device=device,
            dtype=torch.long
        )

        result = model.classify_batch(images)

        probabilities = result["probabilities"]
        predictions = result["predictions"]

        true_probabilities = probabilities[
            torch.arange(
                labels.shape[0],
                device=labels.device
            ),
            labels
        ]

        loss = -torch.log(
            true_probabilities + 1e-8
        ).mean().item()

        total_loss += (
            loss * labels.shape[0]
        )

        total_correct += (
            predictions == labels
        ).sum().item()

        total_images += labels.shape[0]

        total_steps += sum(
            result["inference_steps"]
        )

        total_converged += sum(
            int(value)
            for value in result["converged"]
        )

    return (
        total_loss / total_images,
        100.0 * total_correct / total_images,
        total_steps / total_images,
        100.0 * total_converged / total_images
    )

In [15]:
history = []

best_val_accuracy = -1.0
best_val_loss = float("inf")

best_epoch = None
best_model_state = None
best_k2 = None
best_training_input_count = None

for epoch in range(PC_EPOCHS):
    total_error_0 = 0.0
    total_error_1 = 0.0
    total_error_2 = 0.0
    total_class_error = 0.0

    total_nudged_error_0 = 0.0
    total_nudged_error_1 = 0.0
    total_nudged_error_2 = 0.0

    total_class_loss = 0.0

    total_free_steps = 0
    total_nudge_steps = 0
    total_converged = 0
    total_images = 0

    model.train()

    progress_bar = tqdm(
        train_loader,
        desc=f"Nonlinear Rao PC-CNN Epoch {epoch + 1}/{PC_EPOCHS}"
    )

    for images, labels in progress_bar:
        for image, label in zip(images, labels):
            image = image.to(device)
            label = label.to(
                device=device,
                dtype=torch.long
            )

            result = model(image, label)

            total_error_0 += result["error_0"]
            total_error_1 += result["error_1"]
            total_error_2 += result["error_2"]
            total_class_error += result["class_error"]

            total_nudged_error_0 += result["nudged_error_0"]
            total_nudged_error_1 += result["nudged_error_1"]
            total_nudged_error_2 += result["nudged_error_2"]

            total_class_loss += result["class_loss"]

            total_free_steps += result["inference_steps"]
            total_nudge_steps += result["nudge_steps"]
            total_converged += int(result["converged"])
            total_images += 1

    mean_error_0 = total_error_0 / total_images
    mean_error_1 = total_error_1 / total_images
    mean_error_2 = total_error_2 / total_images
    mean_class_error = total_class_error / total_images

    mean_nudged_error_0 = total_nudged_error_0 / total_images
    mean_nudged_error_1 = total_nudged_error_1 / total_images
    mean_nudged_error_2 = total_nudged_error_2 / total_images

    mean_class_loss = total_class_loss / total_images

    mean_free_steps = total_free_steps / total_images
    mean_nudge_steps = total_nudge_steps / total_images
    training_convergence_rate = (
        100.0 * total_converged / total_images
    )

    train_loss, train_accuracy, train_eval_steps, train_eval_convergence = evaluate_model(
        model,
        train_eval_loader,
        device,
        "Train"
    )

    val_loss, val_accuracy, val_steps, val_convergence = evaluate_model(
        model,
        val_loader,
        device,
        "Validation"
    )

    history.append({
        "epoch": epoch + 1,
        "error_0": mean_error_0,
        "error_1": mean_error_1,
        "error_2": mean_error_2,
        "class_error": mean_class_error,
        "nudged_error_0": mean_nudged_error_0,
        "nudged_error_1": mean_nudged_error_1,
        "nudged_error_2": mean_nudged_error_2,
        "class_loss": mean_class_loss,
        "mean_free_steps": mean_free_steps,
        "mean_nudge_steps": mean_nudge_steps,
        "training_convergence_rate": training_convergence_rate,
        "train_loss": train_loss,
        "train_accuracy": train_accuracy,
        "train_eval_steps": train_eval_steps,
        "train_eval_convergence": train_eval_convergence,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "val_steps": val_steps,
        "val_convergence": val_convergence,
        "k2": model.k2
    })

    is_better = val_loss < best_val_loss

    if (
        val_loss == best_val_loss
        and val_accuracy > best_val_accuracy
    ):
        is_better = True

    if is_better:
        best_val_accuracy = val_accuracy
        best_val_loss = val_loss
        best_epoch = epoch + 1
        best_model_state = copy.deepcopy(
            model.state_dict()
        )
        best_k2 = model.k2
        best_training_input_count = (
            model.training_input_count
        )

    print(
        f"Epoch {epoch + 1}/{PC_EPOCHS} | "
        f"e0: {mean_error_0:.6f} | "
        f"e1: {mean_error_1:.6f} | "
        f"e2: {mean_error_2:.6f} | "
        f"Class e: {mean_class_error:.6f} | "
        f"Nudge e0: {mean_nudged_error_0:.6f} | "
        f"Nudge e1: {mean_nudged_error_1:.6f} | "
        f"Nudge e2: {mean_nudged_error_2:.6f} | "
        f"Class Loss: {mean_class_loss:.4f} | "
        f"Train Acc: {train_accuracy:.2f}% | "
        f"Val Acc: {val_accuracy:.2f}% | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"k2: {model.k2:.8f}"
    )

Nonlinear Rao PC-CNN Epoch 1/5:   0%|          | 0/250 [00:00<?, ?it/s]

Evaluation Nonlinear Rao PC-CNN - Train:   0%|          | 0/250 [00:00<?, ?it/s]

Evaluation Nonlinear Rao PC-CNN - Validation:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 1/5 | e0: 0.070911 | e1: 0.044478 | e2: 0.034898 | Class e: 0.211311 | Nudge e0: 0.067588 | Nudge e1: 0.046513 | Nudge e2: 0.038879 | Class Loss: 2.8711 | Train Acc: 40.62% | Val Acc: 17.90% | Train Loss: 2.1336 | Val Loss: 2.7798 | k2: 0.98522167


Nonlinear Rao PC-CNN Epoch 2/5:   0%|          | 0/250 [00:00<?, ?it/s]

...

Epoch 4/5 | e0: 0.035008 | e1: 0.028467 | e2: 0.030686 | Class e: 0.137212 | Nudge e0: 0.029875 | Nudge e1: 0.027386 | Nudge e2: 0.031813 | Class Loss: 1.1764 | Train Acc: 84.25% | Val Acc: 19.10% | Train Loss: 0.8416 | Val Loss: 2.9750 | k2: 0.91454219


Nonlinear Rao PC-CNN Epoch 5/5:   0%|          | 0/250 [00:00<?, ?it/s]

Evaluation Nonlinear Rao PC-CNN - Train:   0%|          | 0/250 [00:00<?, ?it/s]

Evaluation Nonlinear Rao PC-CNN - Validation:   0%|          | 0/63 [00:00<?, ?it/s]

Epoch 5/5 | e0: 0.034971 | e1: 0.028016 | e2: 0.028999 | Class e: 0.115362 | Nudge e0: 0.029792 | Nudge e1: 0.026865 | Nudge e2: 0.029495 | Class Loss: 0.8938 | Train Acc: 90.30% | Val Acc: 17.60% | Train Loss: 0.6459 | Val Loss: 3.0478 | k2: 0.88771112


In [16]:
trained_u1 = model.level1["conv"].weight.detach().cpu()
trained_u2 = model.level2["conv"].weight.detach().cpu()
trained_u3 = model.level3["conv"].weight.detach().cpu()
trained_wc = model.classifier.weight.detach().cpu()
trained_bc = model.classifier.bias.detach().cpu()

print("AFTER TRAINING")
print("training_input_count:", model.training_input_count)
print("k2:", model.k2)

print("U1 norm:", trained_u1.norm().item())
print("U2 norm:", trained_u2.norm().item())
print("U3 norm:", trained_u3.norm().item())
print("Wc norm:", trained_wc.norm().item())
print("bc:", trained_bc)

print("U1 change:", torch.norm(trained_u1 - initial_u1).item())
print("U2 change:", torch.norm(trained_u2 - initial_u2).item())
print("U3 change:", torch.norm(trained_u3 - initial_u3).item())
print("Wc change:", torch.norm(trained_wc - initial_wc).item())
print("bc change:", torch.norm(trained_bc - initial_bc).item())

AFTER TRAINING
training_input_count: 20000
k2: 0.887711123800987
U1 norm: 1.5222834348678589
U2 norm: 3.86960768699646
U3 norm: 6.574795722961426
Wc norm: 19.930156707763672
bc: tensor([-0.1352,  0.4756,  0.5609, -0.1964,  0.1475,  0.1525, -0.1984, -0.3214,
        -0.3230,  0.2986, -0.6252, -0.6023,  0.1498,  0.6188, -0.3174, -0.0373,
        -0.2429,  0.1699, -0.3191,  0.7453])
U1 change: 1.4187507629394531
U2 change: 3.8365638256073
U3 change: 7.344246864318848
Wc change: 19.151247024536133
bc change: 1.72774076461792


In [17]:
if best_model_state is None:
    raise RuntimeError(
        "No model checkpoint was saved."
    )

model.load_state_dict(best_model_state)
model.k2 = best_k2
model.training_input_count = best_training_input_count
model.eval()

print("Best epoch:", best_epoch)
print(
    "Best validation accuracy:",
    f"{best_val_accuracy:.2f}%"
)
print(
    "Best validation loss:",
    f"{best_val_loss:.4f}"
)

Best epoch: 1
Best validation accuracy: 17.90%
Best validation loss: 2.7798


In [18]:
print("=" * 60)
print("BEST CHECKPOINT DIAGNOSTICS")
print("=" * 60)

print("Best epoch:", best_epoch)
print(
    "training_input_count:",
    model.training_input_count
)
print("k2:", model.k2)

print(
    "U1 norm:",
    model.level1["conv"].weight.norm().item()
)

print(
    "U2 norm:",
    model.level2["conv"].weight.norm().item()
)

print(
    "U3 norm:",
    model.level3["conv"].weight.norm().item()
)

print(
    "Wc norm:",
    model.classifier.weight.norm().item()
)

print(
    "bc:",
    model.classifier.bias.detach().cpu()
)

print(
    "U1 change from initialization:",
    torch.norm(
        model.level1["conv"].weight.detach().cpu()
        - initial_u1
    ).item()
)

print(
    "U2 change from initialization:",
    torch.norm(
        model.level2["conv"].weight.detach().cpu()
        - initial_u2
    ).item()
)

print(
    "U3 change from initialization:",
    torch.norm(
        model.level3["conv"].weight.detach().cpu()
        - initial_u3
    ).item()
)

print(
    "Wc change from initialization:",
    torch.norm(
        model.classifier.weight.detach().cpu()
        - initial_wc
    ).item()
)

print(
    "bc change from initialization:",
    torch.norm(
        model.classifier.bias.detach().cpu()
        - initial_bc
    ).item()
)

BEST CHECKPOINT DIAGNOSTICS
Best epoch: 1
training_input_count: 4000
k2: 0.9852216748768474
U1 norm: 1.453819751739502
U2 norm: 3.3998003005981445
U3 norm: 5.264773845672607
Wc norm: 8.276679992675781
bc: tensor([-0.0184,  0.1596,  0.1885, -0.0528,  0.0484,  0.0524, -0.1132, -0.1201,
        -0.1745,  0.0520, -0.1806, -0.1847,  0.0907,  0.2018, -0.0589, -0.0302,
        -0.0563,  0.0551, -0.0958,  0.2370])
U1 change from initialization: 1.3190070390701294
U2 change from initialization: 3.049229621887207
U3 change from initialization: 4.836756706237793
Wc change from initialization: 6.022233963012695
bc change from initialization: 0.5667500495910645


In [20]:
####################################
@torch.no_grad()
def confidence_diagnostics(model, data_loader, device, name):
    model.eval()
    correct_confidences = []
    wrong_confidences = []
    true_class_probabilities = []

    for images, labels in tqdm(data_loader):
        images = images.to(device)
        labels = labels.to(device=device, dtype=torch.long)
        result = model.classify_batch(images)
        probabilities = result["probabilities"]
        predictions = result["predictions"]
        confidences = probabilities.max(dim=1).values
        true_probs = probabilities[torch.arange(labels.shape[0], device=device), labels]

        correct_mask = predictions == labels
        wrong_mask = predictions != labels

        correct_confidences.extend(confidences[correct_mask].cpu().tolist())
        wrong_confidences.extend(confidences[wrong_mask].cpu().tolist())
        true_class_probabilities.extend(true_probs.cpu().tolist())

    print(name)
    print("Mean true-class probability:", sum(true_class_probabilities) / len(true_class_probabilities))
    print("Mean confidence on correct predictions:", sum(correct_confidences) / len(correct_confidences) if correct_confidences else 0.0)
    print("Mean confidence on wrong predictions:", sum(wrong_confidences) / len(wrong_confidences) if wrong_confidences else 0.0)

# print(f"Train Conv: {train_eval_convergence:.2f}% | Val Conv: {val_convergence:.2f}%")

In [21]:
confidence_diagnostics(model, train_eval_loader, device, "TRAIN")

  0%|          | 0/250 [00:00<?, ?it/s]

TRAIN
Mean true-class probability: 0.21007270770189643
Mean confidence on correct predictions: 0.40081251265910955
Mean confidence on wrong predictions: 0.24397409381991939


In [22]:
confidence_diagnostics(model, val_loader, device, "VALIDATION")

  0%|          | 0/63 [00:00<?, ?it/s]

VALIDATION
Mean true-class probability: 0.12925805854347708
Mean confidence on correct predictions: 0.4434330487134736
Mean confidence on wrong predictions: 0.2375926335186819


In [ ]:
@torch.no_grad()
def evaluate_per_class(
    model,
    data_loader,
    device,
    num_classes
):
    class_correct = torch.zeros(
        num_classes,
        dtype=torch.long
    )

    class_total = torch.zeros(
        num_classes,
        dtype=torch.long
    )

    model.eval()

    for images, labels in data_loader:
        images = images.to(device)

        labels = labels.to(
            device=device,
            dtype=torch.long
        )

        result = model.classify_batch(images)
        predictions = result["predictions"]

        for class_id in range(num_classes):
            mask = labels == class_id

            class_correct[class_id] += (
                predictions[mask]
                == labels[mask]
            ).sum().cpu()

            class_total[class_id] += (
                mask.sum().cpu()
            )

    for class_id in range(num_classes):
        accuracy = (
            100.0
            * class_correct[class_id].item()
            / class_total[class_id].item()
        )

        print(
            f"Class {class_id} | "
            f"Correct: {class_correct[class_id].item()}/"
            f"{class_total[class_id].item()} | "
            f"Accuracy: {accuracy:.2f}%"
        )

evaluate_per_class(
    model,
    val_loader,
    device,
    NUM_CLASSES
)

In [ ]:
with torch.no_grad():
    for class_id in range(NUM_CLASSES):
        print(
            f"Classifier class {class_id} norm:",
            model.classifier.weight[class_id]
            .norm()
            .item()
        )

with torch.no_grad():
    wc = model.classifier.weight.detach()

    similarity = F.cosine_similarity(
        wc.unsqueeze(1),
        wc.unsqueeze(0),
        dim=2
    )

print(
    "Classifier class-weight cosine similarity:"
)
print(similarity.cpu())

In [ ]:
model.eval()

for class_id in selected_classes:
    for image, label in train_subset:
        if int(label) == class_id:
            result = model.classify(
                image.to(device)
            )

            print("True class:", class_id)

            print(
                "Scores:",
                result["scores"]
                .detach()
                .cpu()
                .flatten()
            )

            print(
                "Probabilities:",
                result["probabilities"]
                .detach()
                .cpu()
                .flatten()
            )

            print(
                "Predicted class:",
                result["prediction"].item()
            )

            print(
                "Free inference steps:",
                result["inference_steps"]
            )

            print(
                "Converged:",
                result["converged"]
            )

            print()
            break

In [ ]:
@torch.no_grad()
def create_confusion_matrix(
    model,
    data_loader,
    device,
    num_classes
):
    confusion_matrix = torch.zeros(
        (num_classes, num_classes),
        dtype=torch.long
    )

    model.eval()

    for images, labels in tqdm(
        data_loader,
        desc="Validation Confusion Matrix"
    ):
        images = images.to(device)

        labels = labels.to(
            device=device,
            dtype=torch.long
        )

        result = model.classify_batch(images)
        predictions = result["predictions"]

        for true_label, predicted_label in zip(
            labels,
            predictions
        ):
            confusion_matrix[
                true_label.item(),
                predicted_label.item()
            ] += 1

    return confusion_matrix


val_confusion_matrix = create_confusion_matrix(
    model,
    val_loader,
    device,
    NUM_CLASSES
)

print("Validation Confusion Matrix:")
print(val_confusion_matrix)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

image = ax.imshow(
    val_confusion_matrix.numpy()
)

ax.set_title(
    "Validation Confusion Matrix"
)

ax.set_xlabel(
    "Predicted Class"
)

ax.set_ylabel(
    "True Class"
)

ax.set_xticks(
    range(NUM_CLASSES)
)

ax.set_yticks(
    range(NUM_CLASSES)
)

ax.set_xticklabels(
    [f"Class {i}" for i in range(NUM_CLASSES)],
    rotation=90
)

ax.set_yticklabels(
    [f"Class {i}" for i in range(NUM_CLASSES)]
)

for true_class in range(NUM_CLASSES):
    for predicted_class in range(NUM_CLASSES):
        value = val_confusion_matrix[
            true_class,
            predicted_class
        ].item()

        ax.text(
            predicted_class,
            true_class,
            str(value),
            ha="center",
            va="center",
            fontsize=7
        )

plt.tight_layout()
plt.show()

In [ ]:
saved_bias = model.classifier.bias.detach().clone()

print("CURRENT BIAS")
print(saved_bias.cpu())

original_val_loss, original_val_accuracy, _, _ = evaluate_model(
    model,
    val_loader,
    device,
    "Original Bias"
)

original_confusion_matrix = create_confusion_matrix(
    model,
    val_loader,
    device,
    NUM_CLASSES
)

with torch.no_grad():
    model.classifier.bias.zero_()

zero_bias_val_loss, zero_bias_val_accuracy, _, _ = evaluate_model(
    model,
    val_loader,
    device,
    "Zero Bias"
)

zero_bias_confusion_matrix = create_confusion_matrix(
    model,
    val_loader,
    device,
    NUM_CLASSES
)

print()
print("ORIGINAL BIAS")
print(
    "Validation Loss:",
    f"{original_val_loss:.4f}"
)
print(
    "Validation Accuracy:",
    f"{original_val_accuracy:.2f}%"
)
print(original_confusion_matrix)

print()
print("ZERO BIAS")
print(
    "Validation Loss:",
    f"{zero_bias_val_loss:.4f}"
)
print(
    "Validation Accuracy:",
    f"{zero_bias_val_accuracy:.2f}%"
)
print(zero_bias_confusion_matrix)

with torch.no_grad():
    model.classifier.bias.copy_(saved_bias)

In [ ]:
print(
    "True counts:",
    val_confusion_matrix.sum(dim=1)
)

print(
    "Predicted counts:",
    val_confusion_matrix.sum(dim=0)
)

In [24]:
##############################################
@torch.no_grad()
def collect_class_features(
    model,
    data_loader,
    device,
    num_classes
):
    features_by_class = [
        []
        for _ in range(num_classes)
    ]

    model.eval()

    for images, labels in tqdm(
        data_loader,
        desc="Collecting FREE r3 Features"
    ):
        for image, label in zip(
            images,
            labels
        ):
            image = image.to(device)
            label = int(label)

            _, _, r3, _, _ = model.infer(image)

            features = (
                model.class_features(r3)
                .squeeze(0)
                .detach()
                .cpu()
            )

            features_by_class[label].append(
                features
            )

    features_by_class = [
        torch.stack(features)
        for features in features_by_class
    ]

    return features_by_class


train_features_by_class = collect_class_features(model, train_eval_loader, device, NUM_CLASSES)
val_features_by_class = collect_class_features(model, val_loader, device, NUM_CLASSES)

In [25]:
print("train_features_by_class:", train_features_by_class)
print("val_features_by_class:", val_features_by_class)

train_features_by_class: [tensor([[ 0.0136, -0.0124, -0.0022,  ..., -0.0280,  0.0095, -0.1232],
        [-0.0147, -0.0158, -0.0369,  ...,  0.0086, -0.0110,  0.0262],
        [-0.0784, -0.0037, -0.0081,  ...,  0.1949, -0.0851, -0.2089],
        ...,
        [-0.0227, -0.0547, -0.0386,  ..., -0.1843, -0.2685, -0.1818],
        [ 0.0011,  0.0209,  0.0083,  ..., -0.0433, -0.0069, -0.0115],
        [-0.0891, -0.0537, -0.0366,  ..., -0.0222,  0.0179, -0.2191]]), tensor([[-0.1931, -0.0836, -0.0823,  ..., -0.0531,  0.0311,  0.1758],
        [-0.0568, -0.2478,  0.0322,  ...,  0.0716,  0.0024, -0.0306],
        [-0.1597, -0.1182,  0.0271,  ..., -0.0753,  0.0951, -0.2512],
        ...,
        [-0.1115, -0.1404, -0.1335,  ...,  0.1171,  0.0211,  0.2286],
        [ 0.0796, -0.0773, -0.1689,  ..., -0.2947,  0.0899, -0.1994],
        [-0.0075, -0.1476,  0.0289,  ..., -0.2034, -0.2249,  0.1798]]), tensor([[-0.1015, -0.1499, -0.1265,  ...,  0.0105,  0.0690, -0.2158],
        [-0.0411, -0.0438, -0.0703

In [26]:
@torch.no_grad()
def compare_feature_geometry(train_features_by_class, val_features_by_class):
    train_centroids = torch.stack([features.mean(dim=0) for features in train_features_by_class])
    val_centroids = torch.stack([features.mean(dim=0) for features in val_features_by_class])

    train_spread = torch.tensor([torch.norm(features - train_centroids[class_id], dim=1).mean().item() for class_id, features in enumerate(train_features_by_class)])
    val_spread_to_train = torch.tensor([torch.norm(features - train_centroids[class_id], dim=1).mean().item() for class_id, features in enumerate(val_features_by_class)])
    centroid_shift = torch.norm(val_centroids - train_centroids, dim=1)

    train_centroid_distances = torch.cdist(train_centroids, train_centroids)
    off_diagonal_mask = ~torch.eye(NUM_CLASSES, dtype=torch.bool)
    inter_class_distances = train_centroid_distances[off_diagonal_mask]

    mean_inter_class_distance = inter_class_distances.mean().item()
    minimum_inter_class_distance = inter_class_distances.min().item()
    mean_train_spread = train_spread.mean().item()
    mean_val_spread = val_spread_to_train.mean().item()
    mean_centroid_shift = centroid_shift.mean().item()

    train_separation_ratio = mean_inter_class_distance / mean_train_spread
    val_separation_ratio = mean_inter_class_distance / mean_val_spread

    def nearest_centroid_accuracy(features_by_class, centroids, metric="euclidean"):
        correct = 0
        total = 0
        normalized_centroids = F.normalize(centroids, dim=1)

        for class_id, features in enumerate(features_by_class):
            if metric == "euclidean":
                predictions = torch.cdist(features, centroids).argmin(dim=1)
            else:
                predictions = (F.normalize(features, dim=1) @ normalized_centroids.T).argmax(dim=1)

            correct += (predictions == class_id).sum().item()
            total += features.shape[0]

        return 100.0 * correct / total

    train_nc_euclidean = nearest_centroid_accuracy(train_features_by_class, train_centroids, "euclidean")
    val_nc_euclidean = nearest_centroid_accuracy(val_features_by_class, train_centroids, "euclidean")
    train_nc_cosine = nearest_centroid_accuracy(train_features_by_class, train_centroids, "cosine")
    val_nc_cosine = nearest_centroid_accuracy(val_features_by_class, train_centroids, "cosine")

    print("=" * 78)
    print("FREE r3 TRAIN vs VALIDATION REPRESENTATION DIAGNOSTICS")
    print("=" * 78)
    print(f"Mean TRAIN within-class spread          : {mean_train_spread:.4f}")
    print(f"Mean VAL distance to TRAIN centroid     : {mean_val_spread:.4f}")
    print(f"Mean TRAIN↔VAL centroid shift           : {mean_centroid_shift:.4f}")
    print(f"Mean TRAIN inter-class centroid distance: {mean_inter_class_distance:.4f}")
    print(f"Minimum inter-class centroid distance   : {minimum_inter_class_distance:.4f}")
    print(f"TRAIN separation ratio                  : {train_separation_ratio:.4f}")
    print(f"VALIDATION separation ratio             : {val_separation_ratio:.4f}")
    print("-" * 78)
    print(f"TRAIN nearest-centroid accuracy (Euclidean): {train_nc_euclidean:.2f}%")
    print(f"VAL nearest-centroid accuracy (Euclidean)  : {val_nc_euclidean:.2f}%")
    print(f"TRAIN nearest-centroid accuracy (Cosine)   : {train_nc_cosine:.2f}%")
    print(f"VAL nearest-centroid accuracy (Cosine)     : {val_nc_cosine:.2f}%")
    print("-" * 78)

    for class_id in range(NUM_CLASSES):
        print(f"Class {class_id:02d} | Train Spread: {train_spread[class_id]:.4f} | Val→Train Spread: {val_spread_to_train[class_id]:.4f} | Centroid Shift: {centroid_shift[class_id]:.4f}")

    print("=" * 78)

compare_feature_geometry(train_features_by_class, val_features_by_class)

FREE r3 TRAIN vs VALIDATION REPRESENTATION DIAGNOSTICS
Mean TRAIN within-class spread          : 10.9617
Mean VAL distance to TRAIN centroid     : 10.9494
Mean TRAIN↔VAL centroid shift           : 1.7650
Mean TRAIN inter-class centroid distance: 2.9929
Minimum inter-class centroid distance   : 1.1318
TRAIN separation ratio                  : 0.2730
VALIDATION separation ratio             : 0.2733
------------------------------------------------------------------------------
TRAIN nearest-centroid accuracy (Euclidean): 29.52%
VAL nearest-centroid accuracy (Euclidean)  : 21.00%
TRAIN nearest-centroid accuracy (Cosine)   : 33.30%
VAL nearest-centroid accuracy (Cosine)     : 21.90%
------------------------------------------------------------------------------
Class 00 | Train Spread: 10.3927 | Val→Train Spread: 9.9806 | Centroid Shift: 1.6421
Class 01 | Train Spread: 12.3201 | Val→Train Spread: 12.3437 | Centroid Shift: 2.0395
Class 02 | Train Spread: 10.4272 | Val→Train Spread: 10.6913 | 

In [ ]:
class_centroids = torch.stack([
    features.mean(dim=0)
    for features in val_features_by_class
])

normalized_centroids = F.normalize(
    class_centroids,
    p=2,
    dim=1
)

centroid_similarity = (
    normalized_centroids
    @ normalized_centroids.T
)

print(
    "FREE r3 Class-Centroid Cosine Similarity:"
)
print(centroid_similarity)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

image = ax.imshow(
    centroid_similarity.numpy(),
    vmin=-1,
    vmax=1
)

ax.set_title(
    "FREE r3 Class-Centroid Cosine Similarity"
)

ax.set_xlabel("Class")
ax.set_ylabel("Class")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))

ax.set_xticklabels(
    [f"Class {i}" for i in range(NUM_CLASSES)],
    rotation=90
)

ax.set_yticklabels(
    [f"Class {i}" for i in range(NUM_CLASSES)]
)

plt.colorbar(image)
plt.tight_layout()
plt.show()

In [ ]:
for class_id in range(NUM_CLASSES):
    features = val_features_by_class[class_id]
    centroid = class_centroids[class_id]

    distances = torch.norm(
        features - centroid,
        dim=1
    )

    print(
        f"Class {class_id} | "
        f"Mean distance to centroid: "
        f"{distances.mean().item():.4f} | "
        f"Std: "
        f"{distances.std().item():.4f}"
    )

In [ ]:
global_centroid = torch.cat(
    val_features_by_class,
    dim=0
).mean(dim=0)

centered_centroids = (
    class_centroids - global_centroid
)

normalized_centered_centroids = F.normalize(
    centered_centroids,
    p=2,
    dim=1
)

centered_centroid_similarity = (
    normalized_centered_centroids
    @ normalized_centered_centroids.T
)

print(
    "Centered FREE r3 "
    "Class-Centroid Cosine Similarity:"
)

print(centered_centroid_similarity)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

image = ax.imshow(
    centered_centroid_similarity.numpy(),
    vmin=-1,
    vmax=1
)

ax.set_title(
    "Centered FREE r3 Class-Centroid Cosine Similarity"
)

ax.set_xlabel("Class")
ax.set_ylabel("Class")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))

ax.set_xticklabels(
    [f"Class {i}" for i in range(NUM_CLASSES)],
    rotation=90
)

ax.set_yticklabels(
    [f"Class {i}" for i in range(NUM_CLASSES)]
)

plt.colorbar(image)
plt.tight_layout()
plt.show()

In [ ]:
centroid_distances = torch.cdist(
    class_centroids,
    class_centroids,
    p=2
)

print(
    "FREE r3 Class-Centroid "
    "Euclidean Distance:"
)

print(centroid_distances)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

image = ax.imshow(
    centroid_distances.numpy()
)

ax.set_title(
    "FREE r3 Class-Centroid Euclidean Distance"
)

ax.set_xlabel("Class")
ax.set_ylabel("Class")

ax.set_xticks(range(NUM_CLASSES))
ax.set_yticks(range(NUM_CLASSES))

ax.set_xticklabels(
    [f"Class {i}" for i in range(NUM_CLASSES)],
    rotation=90
)

ax.set_yticklabels(
    [f"Class {i}" for i in range(NUM_CLASSES)]
)

plt.colorbar(image)
plt.tight_layout()
plt.show()

In [ ]:
final_train_loss, final_train_accuracy, final_train_steps, final_train_convergence = evaluate_model(
    model,
    train_eval_loader,
    device,
    "Train"
)

final_val_loss, final_val_accuracy, final_val_steps, final_val_convergence = evaluate_model(
    model,
    val_loader,
    device,
    "Validation"
)

print("RESTORED BEST MODEL")

print(
    "Train Loss:",
    f"{final_train_loss:.4f}"
)

print(
    "Train Accuracy:",
    f"{final_train_accuracy:.2f}%"
)

print(
    "Validation Loss:",
    f"{final_val_loss:.4f}"
)

print(
    "Validation Accuracy:",
    f"{final_val_accuracy:.2f}%"
)

In [ ]:
# Run this cell only after the architecture/hyperparameters are frozen.

test_loss, test_accuracy, test_steps, test_convergence = evaluate_model(
    model,
    test_loader,
    device,
    "Test"
)

print("FINAL CLEAN TEST RESULT")

print(
    "Test Loss:",
    f"{test_loss:.4f}"
)

print(
    "Test Accuracy:",
    f"{test_accuracy:.2f}%"
)

print(
    "Mean Free Inference Steps:",
    f"{test_steps:.2f}"
)

print(
    "Convergence Rate:",
    f"{test_convergence:.2f}%"
)

In [ ]:
def to_display_image(image):
    image = image.detach().cpu()

    base_dataset = getattr(
        train_dataset,
        "base_dataset",
        train_dataset
    )

    mean = getattr(
        base_dataset,
        "mean",
        None
    )

    std = getattr(
        base_dataset,
        "std",
        None
    )

    if mean is not None and std is not None:
        mean_tensor = torch.tensor(
            mean,
            dtype=image.dtype
        ).view(3, 1, 1)

        std_tensor = torch.tensor(
            std,
            dtype=image.dtype
        ).view(3, 1, 1)

        image = (
            image * std_tensor
            + mean_tensor
        )

    image = image.clamp(0, 1)

    return image.permute(
        1, 2, 0
    ).numpy()

In [ ]:
model.eval()

class_examples = {}

for i in range(len(train_subset)):
    image, label = train_subset[i]
    label = int(label)

    if label not in class_examples:
        class_examples[label] = image

    if len(class_examples) == NUM_CLASSES:
        break

fig, axes = plt.subplots(
    NUM_CLASSES,
    2,
    figsize=(10, NUM_CLASSES * 3)
)

for row, class_id in enumerate(
    selected_classes
):
    image = class_examples[class_id]

    result = model.reconstruct(
        image
    )

    reconstruction = result[
        "reconstruction"
    ]

    predicted_class = result[
        "predicted_class"
    ]

    axes[row, 0].imshow(
        to_display_image(image)
    )

    axes[row, 0].set_title(
        f"Original | True Class {class_id}"
    )

    axes[row, 0].axis("off")

    axes[row, 1].imshow(
        to_display_image(reconstruction)
    )

    axes[row, 1].set_title(
        f"Reconstruction | "
        f"Predicted Class {predicted_class}"
    )

    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
epochs_plot = [
    item["epoch"]
    for item in history
]

train_accuracy_plot = [
    item["train_accuracy"]
    for item in history
]

val_accuracy_plot = [
    item["val_accuracy"]
    for item in history
]

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_plot,
    train_accuracy_plot,
    marker="o",
    label="Train Accuracy"
)

plt.plot(
    epochs_plot,
    val_accuracy_plot,
    marker="o",
    label="Validation Accuracy"
)

plt.axhline(
    100.0 / NUM_CLASSES,
    linestyle="--",
    label="Chance Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")

plt.title(
    "Nonlinear Rao PC-CNN "
    "Classification Accuracy"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
train_loss_plot = [
    item["train_loss"]
    for item in history
]

val_loss_plot = [
    item["val_loss"]
    for item in history
]

class_loss_plot = [
    item["class_loss"]
    for item in history
]

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_plot,
    train_loss_plot,
    marker="o",
    label="Train Evaluation Loss"
)

plt.plot(
    epochs_plot,
    val_loss_plot,
    marker="o",
    label="Validation Loss"
)

plt.plot(
    epochs_plot,
    class_loss_plot,
    marker="o",
    label="Online FREE-State Class Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Cross-Entropy Loss")

plt.title(
    "Nonlinear Rao PC-CNN "
    "Classification Loss"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
error_0_plot = [
    item["error_0"]
    for item in history
]

error_1_plot = [
    item["error_1"]
    for item in history
]

error_2_plot = [
    item["error_2"]
    for item in history
]

class_error_plot = [
    item["class_error"]
    for item in history
]

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_plot,
    error_0_plot,
    marker="o",
    label="e0: Image Error"
)

plt.plot(
    epochs_plot,
    error_1_plot,
    marker="o",
    label="e1: r1 Error"
)

plt.plot(
    epochs_plot,
    error_2_plot,
    marker="o",
    label="e2: r2 Error"
)

plt.plot(
    epochs_plot,
    class_error_plot,
    marker="o",
    label="Class Error"
)

plt.xlabel("Epoch")
plt.ylabel("RMSE")

plt.title(
    "FREE-State Prediction Errors"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
nudged_error_0_plot = [
    item["nudged_error_0"]
    for item in history
]

nudged_error_1_plot = [
    item["nudged_error_1"]
    for item in history
]

nudged_error_2_plot = [
    item["nudged_error_2"]
    for item in history
]

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_plot,
    nudged_error_0_plot,
    marker="o",
    label="Nudged e0"
)

plt.plot(
    epochs_plot,
    nudged_error_1_plot,
    marker="o",
    label="Nudged e1"
)

plt.plot(
    epochs_plot,
    nudged_error_2_plot,
    marker="o",
    label="Nudged e2"
)

plt.xlabel("Epoch")
plt.ylabel("RMSE")

plt.title(
    "Supervised-Nudged Predictive Errors"
)

plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
k2_plot = [
    item["k2"]
    for item in history
]

plt.figure(figsize=(8, 5))

plt.plot(
    epochs_plot,
    k2_plot,
    marker="o"
)

plt.xlabel("Epoch")
plt.ylabel("k2")

plt.title(
    "Rao k2 Learning-Rate Schedule "
    "for U1/U2/U3"
)

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
print("=" * 72)

print(
    "NONLINEAR SUPERVISED "
    "HIERARCHICAL RAO PC-CNN V3 "
    "EXPERIMENT SUMMARY"
)

print("=" * 72)

print("Device:", device)
print("Input size:", IMAGE_SIZE)
print("Number of classes:", NUM_CLASSES)

print(
    "Training images:",
    len(train_subset)
)

print(
    "Validation images:",
    len(val_subset)
)

print(
    "Test images:",
    len(test_subset)
)

print(
    "Training epochs:",
    PC_EPOCHS
)

print(
    "Free inference steps:",
    INFERENCE_STEPS
)

print(
    "Supervised nudge steps:",
    SUPERVISED_NUDGE_STEPS
)

print(
    "Convergence tolerance:",
    model.convergence_tolerance
)

print(
    "sigma_class_sq:",
    model.sigma_class_sq
)

print(
    "class_learning_rate:",
    model.class_learning_rate
)

print(
    "Best epoch:",
    best_epoch
)

print(
    "Best validation accuracy:",
    f"{best_val_accuracy:.2f}%"
)

print(
    "Best validation loss:",
    f"{best_val_loss:.4f}"
)

print(
    "Final training accuracy:",
    f"{final_train_accuracy:.2f}%"
)

print(
    "Final validation accuracy:",
    f"{final_val_accuracy:.2f}%"
)

print(
    "Final clean test loss:",
    f"{test_loss:.4f}"
)

print(
    "Final clean test accuracy:",
    f"{test_accuracy:.2f}%"
)

print(
    "Final clean test convergence:",
    f"{test_convergence:.2f}%"
)

print(
    "Training presentations at best epoch:",
    model.training_input_count
)

print(
    "k2 at best epoch:",
    model.k2
)

print("=" * 72)

In [ ]:
# ============================================================
# SIDE-BY-SIDE PARAMETER + TRAINABILITY SUMMARY
# ============================================================

u1_parameters = model.level1["conv"].weight.numel()
u2_parameters = model.level2["conv"].weight.numel()
u3_parameters = model.level3["conv"].weight.numel()

classifier_weights = model.classifier.weight.numel()
classifier_biases = model.classifier.bias.numel()

visual_parameters = u1_parameters + u2_parameters + u3_parameters
classifier_parameters = classifier_weights + classifier_biases
total_parameters = visual_parameters + classifier_parameters

autograd_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)

r1_state_values = model.level1["conv"].out_channels * model.r1_size * model.r1_size
r2_state_values = model.level2["conv"].out_channels * model.r2_size * model.r2_size
r3_state_values = model.level3["conv"].out_channels * model.r3_size * model.r3_size

total_inference_states = r1_state_values + r2_state_values + r3_state_values

print("=" * 120)
print("RAO-STYLE PC-CNN V3 — PARAMETER COUNT AND TRAINABILITY")
print("=" * 120)

print(f"{'Component':<22} {'Count':>14} {'Manual Trainable':>18} {'Autograd':>12} {'Type':>18} {'Learning / Role':<40}")
print("-" * 120)

print(f"{'U1':<22} {u1_parameters:>14,} {'YES':>18} {'OFF':>12} {'Model Parameter':>18} {'Local e0 + r1_nudged update':<40}")
print(f"{'U2':<22} {u2_parameters:>14,} {'YES':>18} {'OFF':>12} {'Model Parameter':>18} {'Local e1 + r2_nudged update':<40}")
print(f"{'U3':<22} {u3_parameters:>14,} {'YES':>18} {'OFF':>12} {'Model Parameter':>18} {'Local e2 + r3_nudged update':<40}")

print(f"{'Classifier W':<22} {classifier_weights:>14,} {'YES':>18} {'OFF':>12} {'Model Parameter':>18} {'Local e_class + FREE r3 update':<40}")
print(f"{'Classifier bias':<22} {classifier_biases:>14,} {'YES':>18} {'OFF':>12} {'Model Parameter':>18} {'Local class-error update':<40}")

print(f"{'deconv1':<22} {0:>14,} {'NO separate':>18} {'OFF':>12} {'Tied Weight':>18} {'Reuses U1':<40}")
print(f"{'deconv2':<22} {0:>14,} {'NO separate':>18} {'OFF':>12} {'Tied Weight':>18} {'Reuses U2':<40}")
print(f"{'deconv3':<22} {0:>14,} {'NO separate':>18} {'OFF':>12} {'Tied Weight':>18} {'Reuses U3':<40}")

print(f"{'r1':<22} {r1_state_values:>14,} {'NO':>18} {'NO':>12} {'Inference State':>18} {'Changes during neural inference':<40}")
print(f"{'r2':<22} {r2_state_values:>14,} {'NO':>18} {'NO':>12} {'Inference State':>18} {'Changes during neural inference':<40}")
print(f"{'r3':<22} {r3_state_values:>14,} {'NO':>18} {'NO':>12} {'Inference State':>18} {'Changes during neural inference':<40}")

print(f"{'tanh':<22} {0:>14,} {'NO':>18} {'NO':>12} {'Function':>18} {'Nonlinear representation':<40}")
print(f"{'Softmax':<22} {0:>14,} {'NO':>18} {'NO':>12} {'Function':>18} {'Converts logits to probabilities':<40}")

print("-" * 120)

print(f"{'Visual parameters':<22} {visual_parameters:>14,}")
print(f"{'Classifier parameters':<22} {classifier_parameters:>14,}")
print(f"{'TOTAL PARAMETERS':<22} {total_parameters:>14,}")
print(f"{'Autograd parameters':<22} {autograd_parameters:>14,}")
print(f"{'Inference state values':<22} {total_inference_states:>14,}")

print("=" * 120)

assert total_parameters == sum(parameter.numel() for parameter in model.parameters())
assert autograd_parameters == 0
assert model.level1["conv"].weight is model.level1["deconv"].weight
assert model.level2["conv"].weight is model.level2["deconv"].weight
assert model.level3["conv"].weight is model.level3["deconv"].weight

print("All parameter and trainability checks passed.")